### 1. 要怎麼作實際的應用

In [1]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = fm.FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)

In [2]:
import numpy as np
import torch

class FeaturesEmbedding(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return self.embedding(x)

class FeaturesLinear(torch.nn.Module):

    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias

class FactorizationMachine(torch.nn.Module):

    def __init__(self, reduce_sum=True):
        super().__init__()
        self.reduce_sum = reduce_sum

    def forward(self, x):
        square_of_sum = torch.sum(x, dim=1) ** 2
        sum_of_square = torch.sum(x ** 2, dim=1)
        ix = square_of_sum - sum_of_square
        if self.reduce_sum:
            ix = torch.sum(ix, dim=1, keepdim=True)
        return 0.5 * ix

class FactorizationMachineModel(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = FeaturesEmbedding(field_dims, embed_dim)
        self.linear = FeaturesLinear(field_dims)
        self.fm = FactorizationMachine(reduce_sum=True)

    def forward(self, x):
        x = self.linear(x) + self.fm(self.embedding(x))
        return x.squeeze(1)

VAR_NUM = 8
K = VAR_NUM
FIELD_DIMS = [2] * VAR_NUM

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)

### 2. 要怎麼檢查 FM 模型實際內部組成

In [3]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
print(model)

FactorizationMachineModel(
  (embedding): FeaturesEmbedding(
    (embedding): Embedding(16, 4)
  )
  (linear): FeaturesLinear(
    (fc): Embedding(16, 1)
  )
  (fm): FactorizationMachine()
)


In [4]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
for name, param in model.named_parameters():
    print(f"{name} : {param.shape}")

embedding.embedding.weight : torch.Size([16, 4])
linear.bias : torch.Size([1])
linear.fc.weight : torch.Size([16, 1])


### 3. 如何將 FM model 作參數的提取

In [5]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
params = model.state_dict()

print(params['embedding.embedding.weight'], "\n")
print(params['linear.bias'], "\n")
print(params['linear.fc.weight'], "\n")

tensor([[-0.0177,  0.2961, -0.1126, -0.1515],
        [ 0.1926, -0.3700, -0.1004,  0.0983],
        [ 0.3283,  0.5270,  0.5393,  0.2423],
        [ 0.3961, -0.5135, -0.3410, -0.4192],
        [ 0.1291,  0.1674,  0.3211,  0.1050],
        [ 0.5429,  0.3009, -0.3217,  0.0095],
        [ 0.0011,  0.5454, -0.1401,  0.1298],
        [-0.5227,  0.1401,  0.3439, -0.2085],
        [-0.0084,  0.2767,  0.0532,  0.0327],
        [-0.0439, -0.2190, -0.3507,  0.3295],
        [-0.5207, -0.3762,  0.0273, -0.0852],
        [ 0.3329,  0.1347, -0.5442, -0.4905],
        [-0.0247, -0.1135,  0.3876, -0.3676],
        [-0.1950,  0.1289,  0.1819,  0.1812],
        [ 0.3777, -0.3590,  0.5116, -0.3270],
        [-0.3333,  0.1365,  0.0440, -0.0843]]) 

tensor([0.]) 

tensor([[-0.9338],
        [-0.6283],
        [-0.2551],
        [ 1.6315],
        [-1.8335],
        [-2.6511],
        [-0.6441],
        [-1.5077],
        [-0.2943],
        [-0.4276],
        [-1.0727],
        [-0.4894],
        [-0.6983],

In [6]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
params = model.state_dict()

torch.set_printoptions(threshold=float('inf'))

print(params['embedding.embedding.weight'], "\n")
print(params['linear.bias'], "\n")
print(params['linear.fc.weight'], "\n")

tensor([[-3.6215e-01,  3.4069e-01, -5.4231e-01,  1.0626e-01],
        [-1.1286e-01,  1.8609e-05, -5.1068e-01,  4.5911e-01],
        [ 2.8631e-01, -4.2321e-01,  2.6955e-01, -3.9420e-01],
        [-1.7345e-01, -2.9614e-01,  3.7667e-01, -4.9786e-01],
        [ 1.3825e-01,  3.9382e-01,  2.3532e-01, -1.7373e-01],
        [-3.6176e-01, -2.6411e-01, -6.1846e-02,  1.3923e-01],
        [ 1.7425e-01, -1.3638e-01,  3.3557e-01,  1.4479e-01],
        [ 7.0608e-02,  3.9648e-01,  3.1352e-01,  3.8795e-01],
        [-5.3074e-01, -2.4939e-01, -2.8300e-01, -4.2388e-02],
        [-1.1891e-01,  2.8014e-01, -4.3393e-01,  1.4461e-01],
        [ 5.1831e-01,  4.5292e-01,  3.4025e-01, -2.0771e-01],
        [ 4.1530e-01,  3.6849e-01, -2.0526e-01,  5.0259e-01],
        [-2.2371e-01,  2.9520e-01,  2.0753e-01, -1.3742e-01],
        [-2.9964e-01,  3.6577e-01, -1.2246e-01, -4.6232e-01],
        [ 4.2347e-01,  2.1942e-01,  2.3765e-02,  3.2159e-02],
        [ 3.0928e-01, -2.2179e-01, -4.3606e-01,  1.5845e-01]]) 

tenso

In [7]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)
params = model.state_dict()
torch.save(model.state_dict(), 'fm_params.pth')

### 4. 要怎麼計算模型內所有參數數量

In [8]:
import FactorizationMachineModel as fm

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

embedding.embedding.weight : 64
linear.bias : 1
linear.fc.weight : 16
Total: 81


### 5. 如果用原始的耦合方式，變數數量會有差異嗎?

In [9]:
import numpy as np
import torch

class FeaturesLinear_01(torch.nn.Module):
    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias


class FactorizationMachine01(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        embed = self.embedding(x)
        num_fields = embed.size(1)
        interaction = []
        for i in range(num_fields):
            for j in range(i + 1, num_fields):
                dot = (embed[:, i, :] * embed[:, j, :]).sum(dim=1)
                interaction.append(dot)

        interaction = torch.stack(interaction, dim=1)
        return interaction.sum(dim=1, keepdim=True)


class FactorizationMachineModel01(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.linear = FeaturesLinear_01(field_dims)
        self.fm = FactorizationMachine01(field_dims, embed_dim)

    def forward(self, x):
        x = self.linear(x) + self.fm(x)
        return x.squeeze(1)

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel01(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

linear.bias : 1
linear.fc.weight : 16
fm.embedding.weight : 64
Total: 81


In [10]:
import numpy as np
import torch

class FeaturesLinear_02(torch.nn.Module):
    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias


class FactorizationMachine02(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        embed = self.embedding(x)
        all_interactions = torch.bmm(embed, embed.transpose(1, 2))
        upper_tri = torch.triu(all_interactions, diagonal=1)
        output = upper_tri.sum(dim=(1, 2), keepdim=True)
        return output.squeeze(2)


class FactorizationMachineModel02(torch.nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.linear = FeaturesLinear_02(field_dims)
        self.fm = FactorizationMachine02(field_dims, embed_dim)

    def forward(self, x):
        x = self.linear(x) + self.fm(x)
        return x.squeeze(1)

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel02(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

linear.bias : 1
linear.fc.weight : 16
fm.embedding.weight : 64
Total: 81


In [11]:
import numpy as np
import torch

class FeaturesEmbedding(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return self.embedding(x)

class FeaturesLinear(torch.nn.Module):

    def __init__(self, field_dims, output_dim=1):
        super().__init__()
        self.fc = torch.nn.Embedding(sum(field_dims), output_dim)
        self.bias = torch.nn.Parameter(torch.zeros((output_dim,)))
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.long)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return torch.sum(self.fc(x), dim=1) + self.bias

class FactorizationMachine03(torch.nn.Module):

    def __init__(self, reduce_sum=True):
        super().__init__()
        self.reduce_sum = reduce_sum

    def forward(self, x):
        square_of_sum = torch.sum(x, dim=1) ** 2
        sum_of_square = torch.sum(x ** 2, dim=1)
        ix = square_of_sum - sum_of_square
        if self.reduce_sum:
            ix = torch.sum(ix, dim=1, keepdim=True)
        return 0.5 * ix

class FactorizationMachineModel03(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = FeaturesEmbedding(field_dims, embed_dim)
        self.linear = FeaturesLinear(field_dims)
        self.fm = FactorizationMachine03(reduce_sum=True)

    def forward(self, x):
        x = self.linear(x) + self.fm(self.embedding(x))
        return x.squeeze(1)

VAR_NUM = 8
K = VAR_NUM
FIELD_DIMS = [2] * VAR_NUM

model = FactorizationMachineModel(field_dims=FIELD_DIMS, embed_dim=K)

VAR_NUM = 8
K = 4
FIELD_DIMS = [2] * 8

model = FactorizationMachineModel03(field_dims=FIELD_DIMS, embed_dim=K)

for name, param in model.named_parameters():
    print(f"{name} : {param.numel()}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {trainable_params}")

embedding.embedding.weight : 64
linear.bias : 1
linear.fc.weight : 16
Total: 81


### 6. 延續上題，做運算時長得比較

In [12]:
import torch
import torch.utils.benchmark as benchmark

VAR_NUM    = 100
K          = 50
FIELD_DIMS = [2] * VAR_NUM
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Use: {device}\n")

model_loop    = FactorizationMachineModel01(FIELD_DIMS, K).to(device).eval()
model_triu    = FactorizationMachineModel02(FIELD_DIMS, K).to(device).eval()
model_formula = FactorizationMachineModel03(FIELD_DIMS, K).to(device).eval()

x = torch.randint(0, 2, (BATCH_SIZE, VAR_NUM), device=device)

timer_loop = benchmark.Timer(
    stmt='model_loop(x)',
    globals={'model_loop': model_loop, 'x': x},
    label='Factorization Machine', 
    sub_label='1. Loop',
    description='Forward Pass'
)

timer_triu = benchmark.Timer(
    stmt='model_triu(x)',
    globals={'model_triu': model_triu, 'x': x},
    label='Factorization Machine', 
    sub_label='2. Triu',
    description='Forward Pass'
)

timer_formula = benchmark.Timer(
    stmt='model_formula(x)',
    globals={'model_formula': model_formula, 'x': x},
    label='Factorization Machine', 
    sub_label='3. Formula',
    description='Forward Pass'
)

results = [
    timer_loop.timeit(number=100),
    timer_triu.timeit(number=100),
    timer_formula.timeit(number=100),
]

benchmark.Compare(results).print()

Use: cuda

[---- Factorization Machine ----]
                  |  Forward Pass
1 threads: ----------------------
      1. Loop     |    216874.3  
      2. Triu     |      1050.1  
      3. Formula  |       749.8  

Times are in microseconds (us).

